# Spam & Phishing Detector — Inference Integration

## Objective

In this notebook, I integrate my trained SMS spam, phishing email, and URL classification models into a reusable inference system.

My goal is to verify that all three classifiers can be loaded and used independently of their training notebooks.

I separate model inference from the application interface so that the same prediction functions can be reused in Streamlit and automated tests.

For URL classification, I retain the Version 3 Histogram Gradient Boosting research baseline and its selected threshold of 0.50.

I also preserve the limitations identified during evaluation. In particular, a URL classified as legitimate by the model must not be interpreted as proof that the destination is safe.

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().resolve().parent

MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"

required_files = {
    "SMS classifier": MODELS_DIR / "sms_svm_pipeline.joblib",
    "Email classifier": MODELS_DIR / "phishing_svm_pipeline.joblib",
    "URL classifier": MODELS_DIR / "url_v3_hgb_model.joblib",
    "URL configuration": MODELS_DIR / "url_v3_model_config.json",
    "URL feature extractor": SRC_DIR / "url_features.py"
}

print("APPLICATION INTEGRATION — FILE VERIFICATION")
print("Project root:", PROJECT_ROOT)

missing_files = []

for name, path in required_files.items():

    if path.is_file():
        print(f"[FOUND] {name}: {path.name}")
    else:
        print(f"[MISSING] {name}: {path.name}")
        missing_files.append(name)

assert not missing_files, (
    f"Missing required files: {missing_files}"
)

print("\nAll required model artifacts: VERIFIED")

In [4]:
config_path = MODELS_DIR / "url_v3_model_config.json"

with open(config_path, "r", encoding="utf-8") as file:
    url_config = json.load(file)

print("URL MODEL CONFIGURATION")

print("Model:", url_config["model_name"])
print("Role:", url_config["model_role"])
print("Threshold:", url_config["decision_threshold"])
print("Feature count:", url_config["feature_count"])

assert url_config["model_role"] == "research_baseline"
assert url_config["feature_count"] == 21
assert url_config["decision_threshold"] == 0.50

print("\nURL configuration: VERIFIED")

URL MODEL CONFIGURATION
Model: Histogram Gradient Boosting
Role: research_baseline
Threshold: 0.5
Feature count: 21

URL configuration: VERIFIED


## Reusable Inference Module Verification

I created a standalone inference module for my SMS, email, and URL classifiers.

I verify that the module imports successfully and that the serialized models load correctly outside their original training notebooks.

This confirms that the prediction functions are available for integration into Streamlit.

In [7]:
import sys
import importlib

# I make the project root available for local imports.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.inference

# I reload the module after saving my latest changes.
importlib.reload(src.inference)

from src.inference import (
    load_models,
    predict_sms,
    predict_email,
    predict_url,
)

print("Inference module imported successfully.")

# I verify that all trained classifiers load.
loaded_models = load_models()

print("\nLOADED CLASSIFIERS")

for detector in ["sms", "email", "url"]:
    print(
        f"{detector.upper()}: "
        f"{type(loaded_models[detector]).__name__}"
    )

print("\nURL configuration:")
print(
    "Threshold:",
    loaded_models["url_config"]["decision_threshold"]
)

Inference module imported successfully.

LOADED CLASSIFIERS
SMS: Pipeline
EMAIL: Pipeline
URL: HistGradientBoostingClassifier

URL configuration:
Threshold: 0.5


## Inference Function Testing

I test my reusable inference functions using representative SMS messages, email content, and URL strings.

My objective is to verify that the classifiers accept valid inputs and return consistent, structured predictions.

I distinguish successful inference from classification correctness. These examples demonstrate that the integration works, but they do not replace my earlier model evaluations.

I also verify that the URL detector returns the saved decision threshold and communicates the limitations of URL-only classification.

In [10]:
# I test the SMS classifier with ordinary and suspicious messages.

sms_examples = {
    "Ordinary message": (
        "Hi, are we still meeting at the library tomorrow?"
    ),
    "Suspicious message": (
        "Congratulations! You have won a cash prize. "
        "Reply WIN now to claim your reward."
    )
}

print("SMS DETECTOR — INFERENCE TEST")

for description, message in sms_examples.items():

    result = predict_sms(message)

    print(f"\n{description}")
    print("Prediction:", result["label"])
    print("Message:", result["message"])

    assert result["detector"] == "sms"
    assert result["prediction"] in {0, 1}
    assert result["label"] in {"legitimate", "spam"}

print("\nSMS inference structure: PASSED")

SMS DETECTOR — INFERENCE TEST

Ordinary message
Prediction: legitimate
Message: No spam was detected by the model.

Suspicious message
Prediction: spam
Message: This message was flagged as potential spam.

SMS inference structure: PASSED


In [12]:
# I test the phishing email classifier.

email_examples = {
    "Ordinary email": (
        "Hello John, please find the meeting agenda attached. "
        "Our project review is scheduled for Thursday at 10 AM."
    ),
    "Suspicious email": (
        "URGENT: Your account has been suspended. "
        "Verify your password immediately using the link "
        "provided to restore access."
    )
}

print("EMAIL DETECTOR — INFERENCE TEST")

for description, message in email_examples.items():

    result = predict_email(message)

    print(f"\n{description}")
    print("Prediction:", result["label"])
    print("Message:", result["message"])

    assert result["detector"] == "email"
    assert result["prediction"] in {0, 1}
    assert result["label"] in {"legitimate", "phishing"}

print("\nEmail inference structure: PASSED")

EMAIL DETECTOR — INFERENCE TEST

Ordinary email
Prediction: legitimate
Message: No phishing was detected by the model.

Suspicious email
Prediction: phishing
Message: This email was flagged as potential phishing.

Email inference structure: PASSED


In [14]:
# I test URL inference without visiting the destinations.

url_examples = {
    "Ordinary URL": (
        "https://www.python.org/"
    ),
    "Suspicious-looking URL": (
        "http://account-security-verify.example.com/"
        "login/confirm?user=12345"
    ),
    "Third-party hosting URL": (
        "https://docs.google.com/forms/d/e/example/viewform"
    )
}

print("URL DETECTOR — INFERENCE TEST")

for description, url in url_examples.items():

    result = predict_url(url)

    print(f"\n{description}")
    print("Prediction:", result["label"])
    print(
        "Phishing score:",
        f"{result['phishing_probability']:.4f}"
    )
    print("Threshold:", result["threshold"])
    print("Message:", result["message"])

    assert result["detector"] == "url"
    assert result["prediction"] in {0, 1}
    assert result["label"] in {"phishing", "not_flagged"}
    assert 0.0 <= result["phishing_probability"] <= 1.0
    assert result["threshold"] == 0.50

    assert result["prediction"] == int(
        result["phishing_probability"] >= result["threshold"]
    )

    assert result["warning"]

print("\nURL inference structure: PASSED")

URL DETECTOR — INFERENCE TEST

Ordinary URL
Prediction: phishing
Phishing score: 0.8524
Threshold: 0.5
Message: This URL was flagged as potentially phishing.

Suspicious-looking URL
Prediction: phishing
Phishing score: 0.9695
Threshold: 0.5
Message: This URL was flagged as potentially phishing.

Third-party hosting URL
Prediction: not_flagged
Phishing score: 0.0167
Threshold: 0.5
Message: This URL was not flagged, but its safety has not been established.

URL inference structure: PASSED


In [16]:
# I verify that the model loader reuses cached objects.

first_load = load_models()
second_load = load_models()

assert first_load is second_load

for detector in ["sms", "email", "url"]:
    assert first_load[detector] is second_load[detector]

print("MODEL CACHING VERIFICATION")
print("Cached model objects reused: PASSED")

print("\nBASIC INFERENCE INTEGRATION: PASSED")

MODEL CACHING VERIFICATION
Cached model objects reused: PASSED

BASIC INFERENCE INTEGRATION: PASSED


## Input Validation and Error Handling

I test how my inference functions respond to invalid and unsupported inputs.

My objective is to ensure that empty messages, oversized text, malformed URLs, and unsupported URL schemes are rejected before reaching the classifiers.

I also verify that invalid inputs produce clear exceptions rather than unexpected prediction results.

These checks help establish a reliable inference interface before integrating the models into Streamlit.

In [21]:
import pandas as pd
# I verify that invalid text inputs are rejected.

invalid_text_cases = [
    ("Empty string", ""),
    ("Whitespace only", "   "),
    ("Non-string input", 12345),
    ("None input", None),
    ("Oversized text", "A" * 100_001)
]

text_validation_results = []

for detector_name, predictor in [
    ("SMS", predict_sms),
    ("Email", predict_email)
]:
    for case_name, value in invalid_text_cases:

        try:
            predictor(value)
            outcome = "FAILED — accepted invalid input"

        except (ValueError, TypeError) as exc:
            outcome = "PASSED"
            print(f"{detector_name} | {case_name}: {exc}")

        text_validation_results.append({
            "detector": detector_name,
            "test_case": case_name,
            "result": outcome
        })

text_validation_df = pd.DataFrame(text_validation_results)

print("\nTEXT VALIDATION SUMMARY")
display(text_validation_df)

assert (text_validation_df["result"] == "PASSED").all()

print("\nSMS and email input validation: PASSED")

SMS | Empty string: SMS message cannot be empty.
SMS | Whitespace only: SMS message cannot be empty.
SMS | Non-string input: SMS message must be a string.
SMS | None input: SMS message must be a string.
SMS | Oversized text: SMS message exceeds the maximum supported length.
Email | Empty string: Email content cannot be empty.
Email | Whitespace only: Email content cannot be empty.
Email | Non-string input: Email content must be a string.
Email | None input: Email content must be a string.
Email | Oversized text: Email content exceeds the maximum supported length.

TEXT VALIDATION SUMMARY


,detector,test_case,result
0,SMS,Empty string,PASSED
1,SMS,Whitespace only,PASSED
2,SMS,Non-string input,PASSED
3,SMS,None input,PASSED
4,SMS,Oversized text,PASSED
5,Email,Empty string,PASSED
6,Email,Whitespace only,PASSED
7,Email,Non-string input,PASSED
8,Email,None input,PASSED
9,Email,Oversized text,PASSED



SMS and email input validation: PASSED


In [23]:
# I verify that unsupported URL inputs are rejected.

invalid_url_cases = {
    "Empty URL": "",
    "Whitespace only": "   ",
    "Missing scheme": "www.example.com",
    "Unsupported FTP scheme": "ftp://example.com/file",
    "JavaScript scheme": "javascript:alert(1)",
    "Missing hostname": "https://",
    "Hostname contains whitespace": "https://example .com",
    "Invalid IPv6 syntax": "https://[invalid",
    "Non-string input": 12345,
    "Oversized URL": "https://example.com/" + "a" * 5000
}

url_validation_results = []

for case_name, value in invalid_url_cases.items():

    try:
        predict_url(value)
        outcome = "FAILED — accepted invalid input"

    except (ValueError, TypeError) as exc:
        outcome = "PASSED"
        print(f"{case_name}: {exc}")

    url_validation_results.append({
        "test_case": case_name,
        "result": outcome
    })

url_validation_df = pd.DataFrame(url_validation_results)

print("\nURL VALIDATION SUMMARY")
display(url_validation_df)

assert (url_validation_df["result"] == "PASSED").all()

print("\nURL input validation: PASSED")

Empty URL: URL cannot be empty.
Whitespace only: URL cannot be empty.
Missing scheme: URL must begin with http:// or https://.
Unsupported FTP scheme: URL must begin with http:// or https://.
JavaScript scheme: URL must begin with http:// or https://.
Missing hostname: URL must contain a hostname.
Hostname contains whitespace: URL hostname cannot contain whitespace.
Invalid IPv6 syntax: URL format is invalid.
Non-string input: URL must be a string.
Oversized URL: URL exceeds the maximum supported length.

URL VALIDATION SUMMARY


,test_case,result
0,Empty URL,PASSED
1,Whitespace only,PASSED
2,Missing scheme,PASSED
3,Unsupported FTP scheme,PASSED
4,JavaScript scheme,PASSED
5,Missing hostname,PASSED
6,Hostname contains whitespace,PASSED
7,Invalid IPv6 syntax,PASSED
8,Non-string input,PASSED
9,Oversized URL,PASSED



URL input validation: PASSED


In [25]:
# I test unusual but valid inputs.

valid_edge_cases = {
    "SMS with punctuation": (
        "Hey!!! Are you coming???"
    ),
    "Email with line breaks": (
        "Hello,\n\nPlease review the document.\n\nRegards"
    ),
    "HTTPS URL with query": (
        "https://example.com/search?q=machine+learning"
    ),
    "HTTP URL": (
        "http://example.com/path"
    )
}

print("VALID EDGE CASE TESTS")

for case_name, value in valid_edge_cases.items():

    if case_name.startswith("SMS"):
        result = predict_sms(value)

    elif case_name.startswith("Email"):
        result = predict_email(value)

    else:
        result = predict_url(value)

    assert result["prediction"] in {0, 1}

    print(f"{case_name}: PASSED")

print("\nValid edge cases: PASSED")

VALID EDGE CASE TESTS
SMS with punctuation: PASSED
Email with line breaks: PASSED
HTTPS URL with query: PASSED
HTTP URL: PASSED

Valid edge cases: PASSED


## Final Inference Consistency Verification

I verify that my reusable inference functions produce the same predictions as the underlying trained classifiers.

For SMS and email detection, I compare the inference results against direct predictions from the saved scikit-learn pipelines.

For URL classification, I independently extract the 21 numerical URL features and apply the saved decision threshold.

I also verify that the URL probability estimate remains consistent with direct model inference.

This final check helps establish that my inference module reproduces the behavior of the trained models before integration into Streamlit.

In [30]:
import numpy as np
# I compare inference functions against direct model predictions.

models = load_models()

text_consistency_cases = [
    (
        "SMS",
        predict_sms,
        models["sms"],
        [
            "Hi, are you coming to class today?",
            "Congratulations! You have won a prize.",
            "Please call me when you get home."
        ]
    ),
    (
        "Email",
        predict_email,
        models["email"],
        [
            "Hello, the meeting is scheduled for tomorrow.",
            "URGENT: Verify your account password immediately.",
            "Please review the attached project report."
        ]
    )
]

for detector_name, predictor, model, examples in text_consistency_cases:

    direct_predictions = model.predict(examples)

    inference_predictions = [
        predictor(text)["prediction"]
        for text in examples
    ]

    np.testing.assert_array_equal(
        direct_predictions,
        inference_predictions
    )

    print(
        f"{detector_name}: "
        f"{len(examples)} predictions matched"
    )

print("\nText inference consistency: PASSED")

SMS: 3 predictions matched
Email: 3 predictions matched

Text inference consistency: PASSED


In [34]:
# I compare reusable URL inference against direct model predictions.
from src.url_features import extract_features_dataframe

url_consistency_examples = [
    "https://www.python.org/",
    "http://example.com/login",
    "https://example.com/search?q=python",
    "https://docs.google.com/forms/d/e/example/viewform",
    "http://account-verify.example.com/login?user=123"
]

url_model = models["url"]
url_config = models["url_config"]

# I extract features directly from the URL strings.
direct_url_features = extract_features_dataframe(
    url_consistency_examples
)

# I preserve the trained feature ordering.
direct_url_features = direct_url_features[
    url_config["feature_names"]
]

direct_probabilities = (
    url_model.predict_proba(
        direct_url_features
    )[:, 1]
)

direct_predictions = (
    direct_probabilities
    >= url_config["decision_threshold"]
).astype(int)

# I obtain predictions through the reusable inference module.
inference_results = [
    predict_url(url)
    for url in url_consistency_examples
]

inference_probabilities = np.array([
    result["phishing_probability"]
    for result in inference_results
])

inference_predictions = np.array([
    result["prediction"]
    for result in inference_results
])

# I verify numerical and classification consistency.
np.testing.assert_allclose(
    direct_probabilities,
    inference_probabilities,
    rtol=1e-12,
    atol=1e-12
)

np.testing.assert_array_equal(
    direct_predictions,
    inference_predictions
)

print("URL INFERENCE CONSISTENCY")

print(
    "URLs tested:",
    len(url_consistency_examples)
)

print("Probability consistency: PASSED")
print("Prediction consistency: PASSED")
print("Feature ordering: PASSED")
print("Threshold consistency: PASSED")

URL INFERENCE CONSISTENCY
URLs tested: 5
Probability consistency: PASSED
Prediction consistency: PASSED
Feature ordering: PASSED
Threshold consistency: PASSED


In [36]:
# I confirm that the inference layer is technically ready for UI integration.

print("FINAL INFERENCE INTEGRATION SUMMARY")
print("-" * 45)

print("SMS model loading: PASSED")
print("Email model loading: PASSED")
print("URL model loading: PASSED")

print("Input validation: PASSED")
print("Model caching: PASSED")

print("Text prediction consistency: PASSED")
print("URL probability consistency: PASSED")
print("URL classification consistency: PASSED")

print("\nPHASE 1 — INFERENCE INTEGRATION: PASSED")

FINAL INFERENCE INTEGRATION SUMMARY
---------------------------------------------
SMS model loading: PASSED
Email model loading: PASSED
URL model loading: PASSED
Input validation: PASSED
Model caching: PASSED
Text prediction consistency: PASSED
URL probability consistency: PASSED
URL classification consistency: PASSED

PHASE 1 — INFERENCE INTEGRATION: PASSED


## Streamlit Application — End-to-End Testing

I test the complete Streamlit application to verify that its three
detection tools behave correctly when accessed through the user interface.

I check input validation, classification results, error handling,
tab navigation, and repeated submissions.

I compare selected UI results against direct inference outputs to
verify that the application uses the intended trained classifiers.

These tests establish application functionality. They do not establish
that the classifiers can reliably identify every spam or phishing attempt.

In [39]:
# I establish reference predictions for comparison with the UI.

reference_cases = {
    "SMS": predict_sms(
        "Congratulations! You have won a cash prize. "
        "Reply WIN now to claim."
    ),
    "Email": predict_email(
        "URGENT: Your account has been suspended. "
        "Verify your password immediately."
    ),
    "URL": predict_url(
        "https://www.python.org/"
    ),
}

print("REFERENCE PREDICTIONS")
print("-" * 45)

for detector, result in reference_cases.items():
    print(f"{detector}: {result['label']}")

    if detector == "URL":
        print(
            "URL phishing score:",
            f"{result['phishing_probability']:.4f}"
        )

REFERENCE PREDICTIONS
---------------------------------------------
SMS: spam
Email: phishing
URL: phishing
URL phishing score: 0.8524
